# EarthCARE Overpasses of ARM SGP — Lidar Cal/Val

Find times the ESA **EarthCARE** satellite passed over the ARM **Southern Great Plains (SGP)** Central Facility, pull the spaceborne **ATLID** lidar for a close overpass, and compare it against the ground-based lidar suite at SGP.

**Workflow**
1. Query the ESA MAAP STAC catalog for EarthCARE ground tracks near SGP
2. Distance-filter overpasses to within 30 km of the site (true geodesic distance to the ground track)
3. Stream the ATLID L1B backscatter for a close overpass
4. Pull the coincident ARM ground-based lidars (Doppler, HSRL, MPL, ceilometer, CL61/CEILPOL) via the ARM Live API
5. Build a cal/val curtain comparison and a quantitative ATLID-vs-HSRL profile match-up

**Colormaps** use [`cmweather`](https://github.com/openradar/cmweather) throughout.

---
### Credentials (environment variables — never hard-code)

This notebook reads two personal tokens from the environment. **Set them in your shell before launching Jupyter; do not paste them into cells.**

```bash
export MAAP_OFFLINE_TOKEN="…"   # ESA MAAP 90-day offline token: https://portal.maap.eo.esa.int/ini/services/auth/token/90dToken.php
export ARM_USER="…"             # ARM Live username
export ARM_TOKEN="…"            # ARM Live API token: https://adc.arm.gov/armlive/
```

- **ESA MAAP** — catalog *discovery* is open; *downloading* data files needs the token. Get a 90-day offline token from the [MAAP portal](https://portal.maap.eo.esa.int/ini/services/auth/token/90dToken.php).
- **ARM Live** — register at [ARM Data Discovery](https://adc.arm.gov/armlive/) for a username + API token.


In [ ]:
import os

# --- Credentials from environment only ---
MAAP_OFFLINE_TOKEN = os.environ["MAAP_OFFLINE_TOKEN"]   # raises KeyError if unset — intentional
ARM_USER  = os.environ["ARM_USER"]
ARM_TOKEN = os.environ["ARM_TOKEN"]

# --- Site: ARM SGP Central Facility (C1), Lamont OK ---
SGP_LAT, SGP_LON = 36.607, -97.488
RADIUS_KM = 30.0

In [ ]:
import numpy as np
import pandas as pd
import requests
import h5py
import xarray as xr
from pystac_client import Client
from shapely.geometry import shape
from pyproj import Geod
from scipy.ndimage import uniform_filter1d

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import cmweather  # registers cmweather colormaps (ChaseSpectral, HomeyerRainbow, ...)

geod = Geod(ellps="WGS84")
CMAP = plt.get_cmap("ChaseSpectral")

## 1. Query EarthCARE overpasses of SGP

The MAAP STAC catalog is queried with `pystac_client`. The **CPR** (Cloud Profiling Radar) nadir product `CPR_NOM_1B` stores its footprint as a `LineString` ground track — ideal for distance-filtering. We search a ~1° box around SGP, then compute the true geodesic minimum distance from each track to the site.

The open collection `EarthCAREL1Validated_MAAP` holds the validated L1 baseline. (Recent acquisitions not yet validated live in `EarthCAREL1InstChecked_MAAP`; future dates in the table are predicted from the repeat orbit.)

In [ ]:
CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue/"
cat = Client.open(CATALOG_URL)

d = 1.0
bbox = [SGP_LON - d, SGP_LAT - d, SGP_LON + d, SGP_LAT + d]

search = cat.search(
    collections=["EarthCAREL1Validated_MAAP"],
    bbox=bbox,
    filter="product:type = 'CPR_NOM_1B'",
    method="GET",
    max_items=5000,
)
items = list(search.items())
print(f"{len(items)} CPR ground-track granules intersect the SGP box")

In [ ]:
def track_min_distance(geom, lat0, lon0, densify=40):
    """Minimum geodesic distance (km) from (lat0, lon0) to a LineString/MultiLineString ground track."""
    lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]
    best = np.inf
    for ln in lines:
        c = list(ln.coords)
        for i in range(len(c) - 1):
            lon1, lat1 = c[i][0], c[i][1]
            lon2, lat2 = c[i + 1][0], c[i + 1][1]
            t = np.linspace(0, 1, densify)
            lo = lon1 + t * (lon2 - lon1)
            la = lat1 + t * (lat2 - lat1)
            _, _, dist = geod.inv(np.full_like(lo, lon0), np.full_like(la, lat0), lo, la)
            best = min(best, dist.min())
    return best / 1000.0


rows = []
for it in items:
    of = it.id.split("_")[-1]              # orbit+frame suffix, e.g. 10540B
    dkm = track_min_distance(shape(it.geometry), SGP_LAT, SGP_LON)
    rows.append({
        "orbit_frame": of, "orbit": of[:-1], "frame": of[-1],
        "start": it.properties.get("start_datetime"),
        "end": it.properties.get("end_datetime"),
        "orbit_state": it.properties.get("sat:orbit_state"),
        "min_dist_km": round(dkm, 2),
        "granule_id": it.id,
    })
df = pd.DataFrame(rows)

# Deduplicate granules reprocessed across baselines (same orbit+frame), keep the closest
df = df.sort_values("min_dist_km").drop_duplicates("orbit_frame", keep="first")
overpasses = df[df.min_dist_km <= RADIUS_KM].sort_values("start").reset_index(drop=True)
print(f"{len(overpasses)} overpasses within {RADIUS_KM:.0f} km of SGP")
overpasses[["start", "orbit", "frame", "orbit_state", "min_dist_km"]].head(12)

## 2. Authenticate with ESA MAAP and stream ATLID

Downloading data files requires a Bearer access token, minted from the personal offline token via an OpenID-Connect refresh-token grant. The `CLIENT_ID`/`CLIENT_SECRET` below are the **public** client credentials published in ESA's MAAP example — the personal offline token (from the environment) is the actual credential.

In [ ]:
def maap_access_token(offline_token):
    """Exchange an ESA MAAP offline token for a short-lived access token.

    CLIENT_ID / CLIENT_SECRET are the PUBLIC client credentials published in ESA's
    MAAP example (identical for every user); the personal offline token is the real
    credential. Overridable via MAAP_CLIENT_ID / MAAP_CLIENT_SECRET if ESA rotates them.
    """
    r = requests.post(
        "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token",
        data={
            "client_id": os.environ.get("MAAP_CLIENT_ID", "offline-token"),
            "client_secret": os.environ.get("MAAP_CLIENT_SECRET", "p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"),
            "grant_type": "refresh_token",
            "refresh_token": offline_token,
            "scope": "offline_access openid",
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["access_token"]


access_token = maap_access_token(MAAP_OFFLINE_TOKEN)
auth = {"Authorization": f"Bearer {access_token}"}
print("MAAP access token acquired")

In [ ]:
# Pick a recent, close overpass and find its ATLID L1B (ATL_NOM_1B) granule.
TARGET_ORBIT_FRAME = "11995D"   # 2026-07-08, descending, ~20 km — change to any row from the table above

atl = list(cat.search(
    collections=["EarthCAREL1Validated_MAAP"], bbox=bbox,
    filter="product:type = 'ATL_NOM_1B'", method="GET", max_items=5000,
).items())
atl_by_of = {it.id.split("_")[-1]: it for it in atl}

it_atl = atl_by_of[TARGET_ORBIT_FRAME]
h5_url = it_atl.assets["enclosure_h5"].href
print("ATLID granule:", it_atl.id)

# Stream to local disk
fn = f"ATL_NOM_1B_{TARGET_ORBIT_FRAME}.h5"
with requests.get(h5_url, headers=auth, stream=True, timeout=600) as r:
    r.raise_for_status()
    with open(fn, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
print("downloaded", round(os.path.getsize(fn) / 1e6, 1), "MB ->", fn)

## 3. Read the ATLID curtain and locate closest approach

ATLID L1B stores measurements under `/ScienceData`. The vertical coordinate is `sample_altitude` (2D, per-profile; the `height` dimension variable is a placeholder). We build a **monotonic** signed along-track distance from cumulative geodesic segment lengths centered at closest approach — the raw point-to-site distance is V-shaped and would scramble `pcolormesh`.

Native single-shot profiles are noisy; a ~15-profile along-track boxcar (≈4.3 km at the ~0.286 km profile spacing) lifts the cloud/aerosol layers, matching the operational quicklook.

In [ ]:
f = h5py.File(fn, "r")
sd = f["/ScienceData"]
slat = sd["sensor_latitude"][:]
slon = sd["sensor_longitude"][:]
_, _, dist = geod.inv(np.full_like(slon, SGP_LON), np.full_like(slat, SGP_LAT), slon, slat)
distkm = dist / 1000.0
i0 = int(np.argmin(distkm))

epoch = np.datetime64("2000-01-01T00:00:00")   # ATLID time is seconds since 2000-01-01
tca = epoch + np.timedelta64(int(sd["time"][i0] * 1000), "ms")
print(f"Closest approach: {tca}  |  {distkm[i0]:.1f} km  |  {slat[i0]:.3f}N {slon[i0]:.3f}E")

In [ ]:
# Context window within 150 km, monotonic signed along-track distance centered at closest approach
ctx = np.where(distkm <= 150)[0]
ca, cb = int(ctx.min()), int(ctx.max())
lonf, latf = slon[ca:cb + 1], slat[ca:cb + 1]
seg = np.zeros(len(lonf))
_, _, dd = geod.inv(lonf[:-1], latf[:-1], lonf[1:], latf[1:])
seg[1:] = np.cumsum(dd) / 1000.0
along = seg - seg[i0 - ca]   # km, 0 at closest approach

mie  = sd["mie_attenuated_backscatter"][ca:cb + 1, :]
salt = sd["sample_altitude"][ca:cb + 1, :]


def along_track_smooth(a, n=15):
    """nan-aware along-track boxcar on positive backscatter, returned as log10."""
    a = a.astype(float); a[a <= 0] = np.nan
    m = np.isfinite(a); a0 = np.where(m, a, 0.0)
    num = uniform_filter1d(a0, n, axis=0, mode="nearest")
    den = uniform_filter1d(m.astype(float), n, axis=0, mode="nearest")
    return np.log10(np.where(den > 0, num / den, np.nan))

mie_s = along_track_smooth(mie)
print("curtain ready:", mie.shape, "| along-track km:", round(along.min()), "to", round(along.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
X = np.broadcast_to(along[:, None], mie.shape)
Y = salt / 1000.0
pm = ax.pcolormesh(X, Y, mie_s, cmap=CMAP, vmin=-7.0, vmax=-4.5, shading="nearest", rasterized=True)
ax.set_ylim(0, 15); ax.set_xlim(-150, 150)
ax.axvline(0, color="k", lw=1.0, ls="--")
ax.axvspan(-RADIUS_KM, RADIUS_KM, color="0.2", alpha=0.08, zorder=0)
ax.set_xlabel("Signed along-track distance from ARM SGP (km)")
ax.set_ylabel("Altitude (km)")
ax.set_title(f"EarthCARE ATLID — Mie co-polar attenuated backscatter\n{str(tca)[:19]} UTC, {distkm[i0]:.1f} km from SGP")
cb = fig.colorbar(pm, ax=ax, pad=0.01); cb.set_label(r"log$_{10}\,\beta$ [1/(m sr)]")
fig.tight_layout()

## 4. Pull coincident ARM ground-based lidars

The ARM Live Data Web Service lists and serves files per datastream. **Note:** the API content-negotiates on `User-Agent` — a default `requests` UA gets HTML, so we send a `curl`-style UA to receive JSON. There is no "list all datastreams" endpoint, so we name the SGP Central Facility (C1) lidar datastreams explicitly.

The SGP **Raman lidar** produced no data in 2026 (retired / long outage) and is omitted.

In [ ]:
ARMLIVE = "https://adc.arm.gov/armlive/livedata"
UA = {"User-Agent": "curl/8.0"}   # force JSON response

# SGP C1 lidar datastreams (instrument : datastream)
DATASTREAMS = {
    "Doppler lidar — vertical stare":   "sgpdlfptC1.b1",
    "Doppler lidar — PPI/RHI scans":    "sgpdlppiC1.b1",
    "Doppler lidar — wind VAP":         "sgpdlprofwind4newsC1.c1",
    "Micropulse lidar — polarized":     "sgpmplpolfsC1.b1",
    "Micropulse lidar — cloud mask":    "sgpmplcmaskmlC1.c1",
    "HSRL":                             "sgphsrlC1.a1",
    "Ceilometer":                       "sgpceilC1.b1",
    "Polarized ceilometer (CL61)":      "sgpceilpolC1.b1",
}
DATE = str(tca)[:10]   # overpass date, YYYY-MM-DD


def arm_list_files(ds, date):
    r = requests.get(f"{ARMLIVE}/query",
                     params={"user": f"{ARM_USER}:{ARM_TOKEN}", "ds": ds, "start": date, "end": date},
                     headers=UA, timeout=60)
    r.raise_for_status()
    ymd = date.replace("-", "")
    return [fn for fn in r.json().get("files", []) if f".{ymd}." in fn]


def arm_download(fn, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    out = os.path.join(dest_dir, fn)
    if not os.path.exists(out) or os.path.getsize(out) == 0:
        r = requests.get(f"{ARMLIVE}/saveData",
                         params={"user": f"{ARM_USER}:{ARM_TOKEN}", "file": fn},
                         headers=UA, timeout=120)
        r.raise_for_status()
        with open(out, "wb") as fo:
            fo.write(r.content)
    return out

In [ ]:
ROOT = f"arm_lidar_{DATE.replace('-', '')}"
manifest = {}
for label, ds in DATASTREAMS.items():
    files = arm_list_files(ds, DATE)
    manifest[ds] = files
    print(f"{label:34s} {ds:26s} {len(files):4d} files")

# Download the curtain instruments used below (stare, HSRL, MPL cloud mask, ceilometer, CL61)
CURTAIN_DS = ["sgpdlfptC1.b1", "sgphsrlC1.a1", "sgpmplcmaskmlC1.c1", "sgpceilC1.b1", "sgpceilpolC1.b1"]
for ds in CURTAIN_DS:
    for fn in manifest[ds]:
        arm_download(fn, f"{ROOT}/{ds}")
    print("downloaded", ds)

## 5. Cal/val comparison — EarthCARE ATLID vs. ground lidars

The spaceborne ATLID transect (spatial, at the overpass instant) is shown above the ground instruments' time–height curtains over a ±90 min window, with the overpass time marked. Each panel keeps its own colorbar (different wavelengths and calibration units).

In [ ]:
tca_ts = pd.Timestamp(str(tca))
w0, w1 = tca_ts - pd.Timedelta("90min"), tca_ts + pd.Timedelta("90min")


def load_concat(ds, keep):
    files = sorted(__import__("glob").glob(f"{ROOT}/{ds}/*"))
    parts = [xr.open_dataset(fp, decode_timedelta=True)[keep] for fp in files]
    return xr.concat(parts, dim="time").sortby("time") if len(parts) > 1 else parts[0]


def timewin(d):
    tt = pd.to_datetime(d.time.values)
    return d.isel(time=np.where((tt >= w0) & (tt <= w1))[0])


dl   = timewin(load_concat("sgpdlfptC1.b1", ["attenuated_backscatter"]))
hsrl = timewin(load_concat("sgphsrlC1.a1", ["atten_beta_a_backscatter"]))
mplc = timewin(load_concat("sgpmplcmaskmlC1.c1", ["range_corrected_backscatter"]))
ceil = timewin(load_concat("sgpceilC1.b1", ["backscatter"]))
cl61 = load_concat("sgpceilpolC1.b1", ["backscatter_p_pol"])   # full day
print("ground data loaded")

In [ ]:
def gmesh(ds, var, hc, hscale):
    t = pd.to_datetime(ds.time.values)
    h = ds[hc].values * hscale
    C = np.log10(np.where(ds[var].values > 0, ds[var].values, np.nan))
    return t, h, C

HMAX = 15.0
fig = plt.figure(figsize=(9.5, 14))
gs = fig.add_gridspec(6, 1, height_ratios=[1.15, 1, 1, 1, 1, 1.1], hspace=0.55)

# Panel 0: ATLID spatial transect
ax0 = fig.add_subplot(gs[0])
pm0 = ax0.pcolormesh(np.broadcast_to(along[:, None], mie.shape), salt / 1000.0, mie_s,
                     cmap=CMAP, vmin=-7.0, vmax=-4.5, shading="nearest", rasterized=True)
ax0.set_ylim(0, HMAX); ax0.set_xlim(-150, 150)
ax0.axvline(0, color="k", lw=1.0, ls="--"); ax0.axvspan(-RADIUS_KM, RADIUS_KM, color="0.2", alpha=0.08, zorder=0)
ax0.set_ylabel("Altitude (km)"); ax0.set_xlabel("Along-track distance from SGP (km)   —   N ◀ ▶ S")
ax0.set_title("EarthCARE ATLID — Mie co-polar attenuated backscatter (spatial transect at overpass)", fontsize=9, loc="left")
fig.colorbar(pm0, ax=ax0, pad=0.01, shrink=0.9).set_label(r"log$_{10}\,\beta$ [1/(m sr)]", fontsize=7)

# Windowed ground panels
panels = [
    ("Doppler lidar (vertical stare)", dl, "attenuated_backscatter", "range", 1e-3, (-6.5, -4.3), r"log$_{10}\,\beta$ [1/(m sr)]"),
    ("HSRL — attenuated aerosol backscatter (532 nm)", hsrl, "atten_beta_a_backscatter", "range", 1e-3, (-7.3, -5.5), r"log$_{10}\,\beta_a$ [1/(m sr)]"),
    ("Micropulse lidar — range-corrected backscatter", mplc, "range_corrected_backscatter", "height", 1.0, (-0.3, 1.3), r"log$_{10}$ RCB [km²·cnt/µs]"),
    ("Ceilometer — attenuated backscatter", ceil, "backscatter", "range", 1e-3, (1.0, 2.9), r"log$_{10}\,\beta$ [1/(sr·km)·10⁻⁴]"),
]
for i, (title, ds, var, hc, hs, (vmn, vmx), clab) in enumerate(panels):
    ax = fig.add_subplot(gs[i + 1])
    t, h, C = gmesh(ds, var, hc, hs)
    pm = ax.pcolormesh(t, h, C.T, cmap=CMAP, vmin=vmn, vmax=vmx, shading="nearest", rasterized=True)
    ax.set_ylim(0, HMAX); ax.set_ylabel("Height AGL (km)"); ax.set_xlim(w0, w1)
    ax.axvline(tca_ts, color="k", lw=1.3, ls="--")
    ax.set_title(title, fontsize=9, loc="left")
    fig.colorbar(pm, ax=ax, pad=0.01, shrink=0.9).set_label(clab, fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M")); ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
    ax.set_xlabel("Time (UTC) — ±90 min window")

# Panel 5: CL61 full day, raw parallel-pol
ax5 = fig.add_subplot(gs[5])
t_all = pd.to_datetime(cl61.time.values); h_all = cl61["range"].values / 1000.0
Zlog = np.log10(np.where(cl61["backscatter_p_pol"].values > 0, cl61["backscatter_p_pol"].values, np.nan))
pm5 = ax5.pcolormesh(t_all, h_all, Zlog.T, cmap=CMAP, vmin=-7.0, vmax=-4.5, shading="nearest", rasterized=True)
ax5.set_ylim(0, HMAX); ax5.set_ylabel("Height AGL (km)")
ax5.axvspan(w0, w1, color="k", alpha=0.10, zorder=3); ax5.axvline(tca_ts, color="k", lw=1.3, ls="--")
ax5.set_xlim(t_all.min(), t_all.max())
ax5.set_title("Polarized ceilometer CL61 (CEILPOL) — raw parallel-pol backscatter, full day (shaded = window above)", fontsize=9, loc="left")
fig.colorbar(pm5, ax=ax5, pad=0.01, shrink=0.9).set_label(r"log$_{10}\,\beta_\parallel$ [1/(m sr)]", fontsize=7)
ax5.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M")); ax5.xaxis.set_major_locator(mdates.HourLocator(interval=3))
ax5.set_xlabel(f"Time (UTC), {DATE} — full day")

fig.suptitle(f"EarthCARE ATLID overpass vs. ground-based lidars at ARM SGP — {DATE}", fontsize=11, y=0.995)
fig.savefig("earthcare_arm_calval.png", dpi=160, bbox_inches="tight")

## 6. Quantitative profile match-up — ATLID vs. HSRL

The single ATLID profile nearest the site (averaged over ±5 km along-track) against the time-matched HSRL profile (±10 min), both as attenuated backscatter in identical units [1/(m sr)] on a common height-above-ground axis.

**Wavelength caveat:** ATLID operates at **355 nm**, the SGP HSRL at **532 nm**. Layer *heights* should coincide, but backscatter *magnitudes* differ through the Ångström exponent — this is a coincidence match-up (~20 km apart, native units), not a cross-calibrated validation.

In [ ]:
# ATLID mean profile within ±5 km along-track, on height-AGL
near = np.where(np.abs(along) <= 5.0)[0]
surf_elev = float(sd["surface_elevation"][i0])
alt_agl = np.nanmean(salt[near, :], axis=0) - surf_elev
mie_prof = np.nanmean(np.where(mie[near, :] > 0, mie[near, :], np.nan), axis=0)

# HSRL mean profile within ±10 min (532 nm, attenuated aerosol backscatter)
tt = pd.to_datetime(hsrl.time.values)
m10 = (tt >= tca_ts - pd.Timedelta("10min")) & (tt <= tca_ts + pd.Timedelta("10min"))
hsrl_beta = hsrl["atten_beta_a_backscatter"].isel(time=np.where(m10)[0]).values
hsrl_prof = np.nanmean(np.where(hsrl_beta > 0, hsrl_beta, np.nan), axis=0)
hsrl_h = hsrl["range"].values

fig, ax = plt.subplots(figsize=(6, 7))
mA = np.isfinite(mie_prof) & (alt_agl >= 0) & (alt_agl <= 15000)
mH = np.isfinite(hsrl_prof) & (hsrl_h >= 0) & (hsrl_h <= 15000)
ax.plot(mie_prof[mA], alt_agl[mA] / 1000, color="#1f77b4", lw=1.6, label="EarthCARE ATLID — Mie co-polar (355 nm)")
ax.plot(hsrl_prof[mH], hsrl_h[mH] / 1000, color="#d62728", lw=1.6, label="ARM SGP HSRL — attenuated aerosol (532 nm)")
ax.set_xscale("log"); ax.set_xlim(1e-8, 1e-4); ax.set_ylim(0, 15)
ax.set_xlabel(r"Attenuated backscatter $\beta$  [1/(m sr)]"); ax.set_ylabel("Height above ground (km)")
ax.set_title(f"Quantitative profile match-up at ARM SGP\nEarthCARE ATLID vs. HSRL — {DATE}", fontsize=9, loc="left")
ax.legend(frameon=False, fontsize=7); ax.grid(True, which="both", lw=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig("earthcare_hsrl_profile.png", dpi=160, bbox_inches="tight")

---

## Notes & caveats

- **Overpass table** reflects the `EarthCAREL1Validated_MAAP` collection. Recent unvalidated acquisitions live in `EarthCAREL1InstChecked_MAAP`; dates beyond "today" are predicted from the repeat orbit.
- **ATLID** is 355 nm; the SGP **HSRL** is 532 nm. The profile match-up compares layer structure, not calibrated magnitudes.
- The comparison is a **coincidence overlay** — the closest approach is ~20 km from the site, and instruments are plotted in native units. A rigorous match-up would interpolate ATLID to the ground track and convert all channels to a common wavelength.
- The SGP **Raman lidar** produced no data in 2026 and is omitted.
- All colormaps are from [`cmweather`](https://github.com/openradar/cmweather).

*Authored with assistance from Claude (Anthropic).*